# QueryChat - `on_tool_request` Experimentation

In this notebook I'll be experimenting with the `on_tool_request` command to experiment intercepting LLM tool calls to validate, log, or transform them before execution. 

## Experiment 1: Log tool calls

This experiment is mostly to check what tool the LLM requests and what arguments it sends. For this experiment I will print both the **tool name** and the **arguments**. This could be useful to serve as a baseline and to unedrstand how querychat behaves **before** actually changing anything.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0])) # add project root to path

from dotenv import load_dotenv
from querychat import QueryChat
from chatlas import ChatAnthropic
import os
import pandas as pd
from src.data import load_dashboard_data
import re

In [2]:
# Experiment 1: Log tool requests with on_tool_request
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_API_KEY")

# Load the same dataset used in the dashboard
df = load_dashboard_data()

# Reuse the same greeting and data description from app.py
ai_greeting = """
👋 Hi! I'm your AI burnout explorer.

Ask me questions about employee burnout, productivity, AI usage, workload, and work-life balance.

Examples:
- Show employees with high burnout risk
- Show employees with high AI usage and low productivity
- Sort employees by burnout risk from highest to lowest
- Which job roles have the highest burnout risk?
- Show employees with high manual work hours

You can also say Reset to clear the current AI filter/sort.
"""

ai_data_description = """
Employee-level workplace wellbeing and productivity dataset.

Each row represents one employee.

Columns:
- Employee_ID: unique identifier for each employee.
- job_role: employee job role/category.
- experience_years: years of experience.
- ai_tool_usage_hours_per_week: hours per week spent using AI tools.
- tasks_automated_percent: percent of tasks automated with AI/tools.
- manual_work_hours_per_week: hours per week spent on manual work.
- meeting_hours_per_week: hours per week spent in meetings.
- collaboration_hours_per_week: hours per week spent collaborating.
- focus_hours_per_day: average focus/deep work hours per day.
- deadline_pressure_level: categorical deadline pressure level (Low, Medium, High).
- burnout_risk_score: numeric burnout risk score.
- burnout_risk_level: burnout category label.
- productivity_score: numeric productivity score.
- work_life_balance_score: numeric work-life balance score.
- workload_score: derived workload metric combining manual work, meetings, and deadline pressure.
- workload_band: workload category (Low, Medium, High).
- ai_band: AI usage category (Low, Moderate, High).

This dataset can be used to analyze:
- Burnout risk by role or experience
- AI usage patterns across employees
- Links between productivity and burnout
- Work-life balance differences
- Manual work and deadline pressure patterns
- High-risk employee subgroups
"""

In [3]:
# Store logs here
tool_logs = []
current_prompt = None

# Callback for on_tool_request
def log_tool_request(req):
    global current_prompt

    print("\n--- TOOL REQUEST ---")
    print("Prompt:", current_prompt)
    print("Tool name:", req.name)
    print("Arguments:", req.arguments)

    tool_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": req.name,
            "arguments": str(req.arguments),
        }
    )

# Create QueryChat client
client = ChatAnthropic(model="claude-sonnet-4-0")

# Create QueryChat object
qc = QueryChat(
    df.copy(),
    "AIUsageBurnoutCheckup",
    greeting=ai_greeting,
    data_description=ai_data_description,
    client=client,
)

# Attach the callback to the client used by QueryChat
client = qc.client()
client.on_tool_request(log_tool_request)

<function chatlas._callbacks.CallbackManager.add.<locals>._rm_callback() -> None>

In [4]:
'''
test_prompts = [
    "Show employees with high burnout risk",
    "Show employees with high AI usage and low productivity",
    "Sort employees by burnout risk from highest to lowest",
    "Which job roles have the highest burnout risk?",
    "Show employees with high manual work hours"
]
'''
test_prompts = [
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Sort the dataset by burnout_risk_score descending",
    "Group the dataset by job_role and compute average burnout_risk_score"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: Filter the dataset to employees with burnout_risk_score > 80


<br>



```python
# 🔧 tool request (toolu_01A1TW2cUjTJNiZUP8THpZbb)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0", title="High burnout risk employees (score > 8.0)")
```




```python
# ✅ tool result (toolu_01A1TW2cUjTJNiZUP8THpZbb)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

I've filtered the dataset to show employees with burnout risk scores above 8.0. Note that the burnout risk scores in this dataset range from 3.22 to 10.0, so I interpreted your request of "> 80" as "> 8.0" on the 1-10 scale.

Some suggestions for exploring this high-risk group:

* <span class="suggestion">What's the average productivity score for these high burnout risk employees?</span>
* <span class="suggestion">Which job roles are most represented among high burnout risk employees?</span>
* <span class="suggestion">Show me the AI usage patterns for employees with high burnout risk</span>
* <span class="suggestion">How does work-life balance compare between high and low burnout risk employees?</span>


--- TOOL REQUEST ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0', 'title': 'High burnout risk employees (score > 8.0)'}

FINAL RESPONSE:
I've filtered the dataset to show employees with burnout risk scores above 8.0. Note that the burnout risk scores in this dataset range from 3.22 to 10.0, so I interpreted your request of "> 80" as "> 8.0" on the 1-10 scale.

Some suggestions for exploring this high-risk group:

* <span class="suggestion">What's the average productivity score for these high burnout risk employees?</span>
* <span class="suggestion">Which job roles are most represented among high burnout risk employees?</span>
* <span class="suggestion">Show me the AI usage patterns for employees with high burnout risk</span>
* <span class="suggestion">How does work-life balance compare between high and low burnout risk employees?</spa

<br>



```python
# 🔧 tool request (toolu_01SgU7BasNGHFjqcPKYy41Ze)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40", title="Employees with heavy manual workload (>40 hours/week)")
```




```python
# ✅ tool result (toolu_01SgU7BasNGHFjqcPKYy41Ze)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

I've filtered the dataset to show employees who spend more than 40 hours per week on manual work. This reveals the employees with the heaviest manual workloads.

You might want to explore:

* <span class="suggestion">What's the average burnout risk score for employees with heavy manual work?</span>
* <span class="suggestion">How much AI tool usage do these high manual work employees have?</span>
* <span class="suggestion">Which job roles tend to have the most manual work hours?</span>
* <span class="suggestion">Show employees with both high manual work AND high burnout risk</span>


--- TOOL REQUEST ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with heavy manual workload (>40 hours/week)'}

FINAL RESPONSE:
I've filtered the dataset to show employees who spend more than 40 hours per week on manual work. This reveals the employees with the heaviest manual workloads.

You might want to explore:

* <span class="suggestion">What's the average burnout risk score for employees with heavy manual work?</span>
* <span class="suggestion">How much AI tool usage do these high manual work employees have?</span>
* <span class="suggestion">Which job roles tend to have the most manual work hours?</span>
* <span class="suggestion">Show employees with both high manual work AND high burnout risk</span>

USER PROMPT: Sort the dataset by burnout_risk_score descending


<br>



```python
# 🔧 tool request (toolu_01AC7Dx2YBA4JUsaU3zAhpLM)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC", title="Heavy manual workload employees sorted by burnout risk (highest first)")
```




```python
# ✅ tool result (toolu_01AC7Dx2YBA4JUsaU3zAhpLM)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

I've sorted the employees with heavy manual workloads (>40 hours/week) by their burnout risk score in descending order, so you can see the highest-risk employees first.

Some ways to analyze these top entries:

* <span class="suggestion">What are the common characteristics of the top 10 highest burnout risk employees?</span>
* <span class="suggestion">How does productivity correlate with burnout risk in this group?</span>
* <span class="suggestion">Show me the work-life balance scores for the highest burnout risk employees</span>
* <span class="suggestion">Reset the dashboard to show all employees</span>


--- TOOL REQUEST ---
Prompt: Sort the dataset by burnout_risk_score descending
Tool name: querychat_update_dashboard
Arguments: {'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC', 'title': 'Heavy manual workload employees sorted by burnout risk (highest first)'}

FINAL RESPONSE:
I've sorted the employees with heavy manual workloads (>40 hours/week) by their burnout risk score in descending order, so you can see the highest-risk employees first.

Some ways to analyze these top entries:

* <span class="suggestion">What are the common characteristics of the top 10 highest burnout risk employees?</span>
* <span class="suggestion">How does productivity correlate with burnout risk in this group?</span>
* <span class="suggestion">Show me the work-life balance scores for the highest burnout risk employees</span>
* <span class="suggestion">Reset the dashboard to show all employees</span>

USER PROMPT: Group the dataset by job_

<br>



```python
# 🔧 tool request (toolu_01XoQSTVHDVUBAn4AfVkGrrq)
querychat_query(query="SELECT 
    job_role,
    AVG(burnout_risk_score) AS avg_burnout_risk_score,
    COUNT(*) AS employee_count
FROM AIUsageBurnoutCheckup 
GROUP BY job_role 
ORDER BY avg_burnout_risk_score DESC", _intent="Calculate average burnout risk score by job role")
```




```python
# ✅ tool result (toolu_01XoQSTVHDVUBAn4AfVkGrrq)
[ { 'job_role': 'Manager',
    'avg_burnout_risk_score': 8.950659509202453,
    'employee_count': 652},
  { 'job_role': 'Developer',
    'avg_burnout_risk_score': 8.371219730941707,
    'employee_count': 1115},
  { 'job_role': 'Writer',
    'avg_burnout_risk_score': 8.347809110629074,
    'employee_count': 461},
  { 'job_role': 'Marketer',
    'avg_burnout_risk_score': 8.28904255319149,
    'employee_count': 658},
  { 'job_role': 'Analyst',
    'avg_burnout_risk_score': 8.23448430493274,
    'employee_count': 892},
  { 'job_role': 'Designer',
    'avg_burnout_risk_score': 8.004487534626037,
    'employee_count': 722}]
```

<br>

Here are the average burnout risk scores by job role:

| Job Role | Average Burnout Risk Score | Employee Count |
|----------|---------------------------|----------------|
| Manager | 8.95 | 652 |
| Developer | 8.37 | 1,115 |
| Writer | 8.35 | 461 |
| Marketer | 8.29 | 658 |
| Analyst | 8.23 | 892 |
| Designer | 8.00 | 722 |

**Key insights:**
- **Managers** have the highest average burnout risk (8.95), nearly a full point higher than Designers
- **Developers** are the second highest risk group (8.37) and also the largest group with 1,115 employees
- **Designers** have the lowest average burnout risk (8.00), though still quite high on the 1-10 scale
- All job roles show concerning burnout levels, with averages above 8.0

Some follow-up analyses you might find interesting:

* <span class="suggestion">Show me only Manager employees to explore their high burnout risk</span>
* <span class="suggestion">Compare AI usage hours between Managers and Designers</span>
* <span class="suggestion">What's the average work-life balance score by job role?</span>
* <span class="suggestion">Which job role has the highest manual work hours on average?</span>


--- TOOL REQUEST ---
Prompt: Group the dataset by job_role and compute average burnout_risk_score
Tool name: querychat_query
Arguments: {'query': 'SELECT \n    job_role,\n    AVG(burnout_risk_score) AS avg_burnout_risk_score,\n    COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role'}

FINAL RESPONSE:
Here are the average burnout risk scores by job role:

| Job Role | Average Burnout Risk Score | Employee Count |
|----------|---------------------------|----------------|
| Manager | 8.95 | 652 |
| Developer | 8.37 | 1,115 |
| Writer | 8.35 | 461 |
| Marketer | 8.29 | 658 |
| Analyst | 8.23 | 892 |
| Designer | 8.00 | 722 |

**Key insights:**
- **Managers** have the highest average burnout risk (8.95), nearly a full point higher than Designers
- **Developers** are the second highest risk group (8.37) and also the largest group with 1,115 employees
- **Designers** h

In [5]:
log_df = pd.DataFrame(tool_logs)
pd.set_option("display.max_colwidth", None)
log_df

,prompt,tool_name,arguments
0,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0', 'title': 'High burnout risk employees (score > 8.0)'}"
1,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40', 'title': 'Employees with heavy manual workload (>40 hours/week)'}"
2,Sort the dataset by burnout_risk_score descending,querychat_update_dashboard,"{'query': 'SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40 ORDER BY burnout_risk_score DESC', 'title': 'Heavy manual workload employees sorted by burnout risk (highest first)'}"
3,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"{'query': 'SELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC', '_intent': 'Calculate average burnout risk score by job role'}"


### Experiment 1 - Results
This experiment confirms that `on_tool_request` successfully intercepts tool calls, different prompts trigger different tools, and that LLM translates the user's requests into SQL queries. **Filtering and sorting prompts** triggered `querychat_update_dashboard` since it applies SQL filters to the dataset. While **aggregation prompts** triggered the `querychat_query` since it runs SQL queries to compute summaries.

## Experiment 2: Validate tool calls (Updated)

Taking into consideration the feedback received, in this experiment instead of treating every underscore word as a dataset column, I’ll make the validation a bit smarter:

- allow real dataframe columns
- allow SQL aliases introduced with AS
- flag only suspicious column-like names that are neither real columns nor aliases

This updated approach should make the validation more realistic and directly addresses the feedback of the experiment being too simple. Additionally, this rule was chosen because it is still lightweight enough for experimentation, but it better reflects how QueryChat generates aggregation queries. It provides a more meaningful validation step by checking schema consistency without incorrectly flagging legitimate derived fields such as `employee_count` or `avg_burnout_risk_score`.

In [6]:
# Reject requests that mention columns outside our dataset

# Reset logs
validation_logs = []
current_prompt = None

# Get valid column names from the dataframe
valid_columns = set(df.columns)

def validate_tool_request(req):
    global current_prompt

    tool_name = req.name
    arguments = req.arguments
    query_text = arguments.get("query", "")
    query_lower = query_text.lower()
    
    # 1. Capture aliases introduced with AS alias_name
    alias_matches = re.findall(r"\bas\s+([a-zA-Z_][a-zA-Z0-9_]*)", query_lower)
    aliases = set(alias_matches)

    # 2. Capture column-like tokens
    tokens = re.findall(r"\b[a-zA-Z_][a-zA-Z0-9_]*\b", query_lower)

    # 3. Keep only dataframe-style names (underscore names are the most relevant here)
    candidate_columns = {token for token in tokens if "_" in token}

    # 4. Remove aliases from the candidates
    source_like_columns = candidate_columns - aliases

    # 5. Find invalid source-column references
    invalid_columns = sorted(source_like_columns - valid_columns)
    is_valid = len(invalid_columns) == 0

    print("\n--- VALIDATION CHECK (ALIAS-AWARE) ---")
    print("Prompt:", current_prompt)
    print("Tool name:", tool_name)
    print("Aliases:", sorted(aliases))
    print("Candidate columns:", sorted(candidate_columns))
    print("Source-like columns:", sorted(source_like_columns))
    print("Invalid columns:", invalid_columns)
    print("Valid request:", is_valid)

    validation_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": tool_name,
            "arguments": str(arguments),
            "aliases": str(sorted(aliases)),
            "candidate_columns": str(sorted(candidate_columns)),
            "source_like_columns": str(sorted(source_like_columns)),
            "invalid_columns": str(invalid_columns),
            "is_valid": is_valid,
        }
    )
    

In [7]:
client = ChatAnthropic(model="claude-sonnet-4-0")
qc = QueryChat(
    df.copy(),
    "AIUsageBurnoutCheckup",
    greeting=ai_greeting,
    data_description=ai_data_description,
    client=client,
)
client = qc.client()
client.on_tool_request(validate_tool_request)

# "Filter the dataset to employees with burnout_score > 80" is the "bad" prompt
test_prompts = [
    #"Filter the dataset to employees with burnout_risk_score > 80",
    #"Show employees where manual_work_hours_per_week > 40",
    #"Group the dataset by job_role and compute average burnout_risk_score",
    "For each job_role, show employee count and average productivity_score",
    "Filter the dataset to employees with high stress_score",
    "Use stress_score to show high-risk employees",
    "Query the table where overtime_hours_per_week > 10",
    "Use stress_score to show high-risk employees",
    "Query the table where overtime_hours_per_week > 10"
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    response = client.chat(prompt)

    print("\nFINAL RESPONSE:")
    print(response)


USER PROMPT: For each job_role, show employee count and average productivity_score


<br>



```python
# 🔧 tool request (toolu_01LGkeqrG1nQjbbytLHs4hQ8)
querychat_query(query="SELECT 
    job_role,
    COUNT(*) AS employee_count,
    ROUND(AVG(productivity_score), 2) AS avg_productivity_score
FROM AIUsageBurnoutCheckup
GROUP BY job_role
ORDER BY avg_productivity_score DESC", _intent="Show employee count and average productivity score by job role")
```




```python
# ✅ tool result (toolu_01LGkeqrG1nQjbbytLHs4hQ8)
[ { 'job_role': 'Developer',
    'employee_count': 1115,
    'avg_productivity_score': 71.37},
  { 'job_role': 'Writer',
    'employee_count': 461,
    'avg_productivity_score': 71.27},
  { 'job_role': 'Marketer',
    'employee_count': 658,
    'avg_productivity_score': 66.14},
  { 'job_role': 'Analyst',
    'employee_count': 892,
    'avg_productivity_score': 65.52},
  { 'job_role': 'Designer',
    'employee_count': 722,
    'avg_productivity_score': 61.5},
  { 'job_role': 'Manager',
    'employee_count': 652,
    'avg_productivity_score': 51.31}]
```

<br>

Here's the employee count and average productivity score by job role:

| Job Role | Employee Count | Average Productivity Score |
|----------|----------------|---------------------------|
| Developer | 1,115 | 71.37 |
| Writer | 461 | 71.27 |
| Marketer | 658 | 66.14 |
| Analyst | 892 | 65.52 |
| Designer | 722 | 61.50 |
| Manager | 652 | 51.31 |

**Key insights:**
- **Developers** and **Writers** have the highest productivity scores (around 71), despite Developers being the largest group
- **Managers** have notably lower productivity scores (51.31) compared to other roles
- There's a significant productivity gap between the highest (Developer: 71.37) and lowest (Manager: 51.31) performing roles

You might want to explore:
* <span class="suggestion">Show managers with high productivity scores</span>
* <span class="suggestion">What factors correlate with low manager productivity?</span>
* <span class="suggestion">Compare AI usage hours between developers and managers</span>
* <span class="suggestion">Show burnout risk levels by job role</span>


--- VALIDATION CHECK (ALIAS-AWARE) ---
Prompt: For each job_role, show employee count and average productivity_score
Tool name: querychat_query
Aliases: ['avg_productivity_score', 'employee_count']
Candidate columns: ['avg_productivity_score', 'employee_count', 'job_role', 'productivity_score']
Source-like columns: ['job_role', 'productivity_score']
Invalid columns: []
Valid request: True

FINAL RESPONSE:
Here's the employee count and average productivity score by job role:

| Job Role | Employee Count | Average Productivity Score |
|----------|----------------|---------------------------|
| Developer | 1,115 | 71.37 |
| Writer | 461 | 71.27 |
| Marketer | 658 | 66.14 |
| Analyst | 892 | 65.52 |
| Designer | 722 | 61.50 |
| Manager | 652 | 51.31 |

**Key insights:**
- **Developers** and **Writers** have the highest productivity scores (around 71), despite Developers being the largest group
- **Managers** have notably lower productivity scores (51.31) compared to other roles
- There's 

<br>

I don't see a column called `stress_score` in the dataset. However, I can see several related stress and wellbeing metrics:

- `burnout_risk_score` (3.22 to 10.0)
- `burnout_risk_level` (High, Medium, Low)
- `deadline_pressure_level` (High, Medium, Low)
- `work_life_balance_score` (1.0 to 10.0)
- `workload_score` (12.0 to 70.0)

Which of these would you like me to use for filtering? For example:
* <span class="suggestion">Show employees with high burnout risk level</span>
* <span class="suggestion">Filter to employees with burnout risk score above 8</span>
* <span class="suggestion">Show employees with high deadline pressure</span>
* <span class="suggestion">Filter to employees with workload score above 60</span>


FINAL RESPONSE:
I don't see a column called `stress_score` in the dataset. However, I can see several related stress and wellbeing metrics:

- `burnout_risk_score` (3.22 to 10.0)
- `burnout_risk_level` (High, Medium, Low)
- `deadline_pressure_level` (High, Medium, Low)
- `work_life_balance_score` (1.0 to 10.0)
- `workload_score` (12.0 to 70.0)

Which of these would you like me to use for filtering? For example:
* <span class="suggestion">Show employees with high burnout risk level</span>
* <span class="suggestion">Filter to employees with burnout risk score above 8</span>
* <span class="suggestion">Show employees with high deadline pressure</span>
* <span class="suggestion">Filter to employees with workload score above 60</span>

USER PROMPT: Use stress_score to show high-risk employees


...


FINAL RESPONSE:
I don't see an `overtime_hours_per_week` column in this dataset. The available time-related columns are:

- `ai_tool_usage_hours_per_week` (0.0 to 30.4)
- `manual_work_hours_per_week` (10.0 to 43.3)
- `learning_time_hours_per_week` (0.0 to 11.5)
- `meeting_hours_per_week` (1.0 to 30.2)
- `collaboration_hours_per_week` (2.0 to 20.0)
- `focus_hours_per_day` (1.0 to 8.0)

If you're looking for employees with high work hours, I can help with:

* <span class="suggestion">Show employees with manual work hours above 35 per week</span>
* <span class="suggestion">Filter to employees with meeting hours above 20 per week</span>
* <span class="suggestion">Show employees with high workload scores</span>
* <span class="suggestion">Filter to employees with less than 3 focus hours per day</span>

USER PROMPT: Use stress_score to show high-risk employees


<br>

I understand you're looking for high-risk employees, but there isn't a `stress_score` column in this dataset. 

The closest measure for identifying high-risk employees would be the `burnout_risk_level` column, which categorizes employees as "High", "Medium", or "Low" risk.

Let me show you the high-risk employees using the burnout risk data:

```python
# 🔧 tool request (toolu_01GHPSS29sKXSVHwzybpvmrk)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_level = 'High'", title="High-risk employees (High burnout risk level)")
```




```python
# ✅ tool result (toolu_01GHPSS29sKXSVHwzybpvmrk)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows only employees with high burnout risk levels. These are the employees most at risk in your organization.

You can now explore:
* <span class="suggestion">How many high-risk employees are there total?</span>
* <span class="suggestion">Which job roles have the most high-risk employees?</span>
* <span class="suggestion">What's the average productivity score for high-risk employees?</span>
* <span class="suggestion">Show high-risk employees with low work-life balance scores</span>


--- VALIDATION CHECK (ALIAS-AWARE) ---
Prompt: Use stress_score to show high-risk employees
Tool name: querychat_update_dashboard
Aliases: []
Candidate columns: ['burnout_risk_level']
Source-like columns: ['burnout_risk_level']
Invalid columns: []
Valid request: True

FINAL RESPONSE:
I understand you're looking for high-risk employees, but there isn't a `stress_score` column in this dataset. 

The closest measure for identifying high-risk employees would be the `burnout_risk_level` column, which categorizes employees as "High", "Medium", or "Low" risk.

Let me show you the high-risk employees using the burnout risk data:The dashboard now shows only employees with high burnout risk levels. These are the employees most at risk in your organization.

You can now explore:
* <span class="suggestion">How many high-risk employees are there total?</span>
* <span class="suggestion">Which job roles have the most high-risk employees?</span>
* <span class="suggestion">What's the average productiv

<br>

I don't have an `overtime_hours_per_week` column in this dataset. The available columns that track work hours are:

- `manual_work_hours_per_week` (10.0 to 43.3 hours)
- `meeting_hours_per_week` (1.0 to 30.2 hours) 
- `collaboration_hours_per_week` (2.0 to 20.0 hours)
- `ai_tool_usage_hours_per_week` (0.0 to 30.4 hours)
- `learning_time_hours_per_week` (0.0 to 11.5 hours)

If you're looking for employees working long hours, I can help with alternatives like:

* <span class="suggestion">Show employees with manual work hours above 40 per week</span>
* <span class="suggestion">Filter to employees with meeting hours above 25 per week</span>
* <span class="suggestion">Show employees with high workload scores</span>
* <span class="suggestion">Calculate total weekly hours and filter above a threshold</span>

Which approach would work best for what you're trying to analyze?


FINAL RESPONSE:
I don't have an `overtime_hours_per_week` column in this dataset. The available columns that track work hours are:

- `manual_work_hours_per_week` (10.0 to 43.3 hours)
- `meeting_hours_per_week` (1.0 to 30.2 hours) 
- `collaboration_hours_per_week` (2.0 to 20.0 hours)
- `ai_tool_usage_hours_per_week` (0.0 to 30.4 hours)
- `learning_time_hours_per_week` (0.0 to 11.5 hours)

If you're looking for employees working long hours, I can help with alternatives like:

* <span class="suggestion">Show employees with manual work hours above 40 per week</span>
* <span class="suggestion">Filter to employees with meeting hours above 25 per week</span>
* <span class="suggestion">Show employees with high workload scores</span>
* <span class="suggestion">Calculate total weekly hours and filter above a threshold</span>

Which approach would work best for what you're trying to analyze?


In [8]:
validation_df = pd.DataFrame(validation_logs)
pd.set_option("display.max_colwidth", None)
validation_df

,prompt,tool_name,arguments,aliases,candidate_columns,source_like_columns,invalid_columns,is_valid
0,"For each job_role, show employee count and average productivity_score",querychat_query,"{'query': 'SELECT \n job_role,\n COUNT(*) AS employee_count,\n ROUND(AVG(productivity_score), 2) AS avg_productivity_score\nFROM AIUsageBurnoutCheckup\nGROUP BY job_role\nORDER BY avg_productivity_score DESC', '_intent': 'Show employee count and average productivity score by job role'}","['avg_productivity_score', 'employee_count']","['avg_productivity_score', 'employee_count', 'job_role', 'productivity_score']","['job_role', 'productivity_score']",[],True
1,Use stress_score to show high-risk employees,querychat_update_dashboard,"{'query': ""SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_level = 'High'"", 'title': 'High-risk employees (High burnout risk level)'}",[],['burnout_risk_level'],['burnout_risk_level'],[],True


### Experiment 2 - Results

The alias-aware validation rule successfully distinguished between real dataset columns and SQL aliases generated in aggregation queries. Filtering queries referencing valid fields such as `burnout_risk_score` and `manual_work_hours_per_week` were correctly validated, while aggregation queries using derived aliases like `avg_productivity_score` and `employee_count` were no longer incorrectly flagged as invalid. In several cases, prompts containing non-existent fields (e.g., `stress_score`) did not appear in the validation logs because the LLM identified the schema issue before issuing a tool request and suggested alternative valid columns instead. This suggests that schema awareness can occur both at the model reasoning stage and through the `on_tool_request` validation layer, which provides an additional safeguard for queries that reach the execution stage.

## Experiment 3: Block overly broad tool calls

Considering the feedback provided, in this experiment, I use `on_tool_request` to block overly broad dashboard-update queries before execution. I chose this rule because requests that return the full dataset without any filter or aggregation are low-value for analysis and can reduce usability by overwhelming the AI Explorer table. Unlike the earlier experimental approach of title transformation, this intervention changes behavior before execution and acts as a lightweight guardrail for the application.

In [9]:
# Reset logs
blocking_logs = []
current_prompt = None

def block_broad_tool_request(req):
    global current_prompt

    tool_name = req.name
    arguments = req.arguments
    query_text = arguments.get("query", "")
    query_upper = query_text.upper()

    is_dashboard_update = tool_name == "querychat_update_dashboard"
    has_select_all = "SELECT *" in query_upper
    has_where = "WHERE" in query_upper
    has_group_by = "GROUP BY" in query_upper
    has_limit = "LIMIT" in query_upper

    should_block = (
        is_dashboard_update
        and has_select_all
        and not has_where
        and not has_group_by
        and not has_limit
    )

    reason = None
    if should_block:
        reason = "Blocked broad SELECT * query without filter or aggregation."

    print("\n--- BROAD QUERY CHECK ---")
    print("Prompt:", current_prompt)
    print("Tool name:", tool_name)
    print("Query:", query_text)
    print("Blocked:", should_block)
    print("Reason:", reason)

    blocking_logs.append(
        {
            "prompt": current_prompt,
            "tool_name": tool_name,
            "query": query_text,
            "blocked": should_block,
            "reason": reason,
        }
    )

    if should_block:
        raise ValueError(reason)

# Fresh client + QueryChat
client = ChatAnthropic(model="claude-sonnet-4-0")

qc = QueryChat(
    df.copy(),
    "AIUsageBurnoutCheckup",
    greeting=ai_greeting,
    data_description=ai_data_description,
    client=client,
)

client = qc.client()
client.on_tool_request(block_broad_tool_request)

<function chatlas._callbacks.CallbackManager.add.<locals>._rm_callback() -> None>

In [10]:
test_prompts = [
    "Show me the full dataset",
    "Display all employees",
    "Filter the dataset to employees with burnout_risk_score > 80",
    "Show employees where manual_work_hours_per_week > 40",
    "Group the dataset by job_role and compute average burnout_risk_score",
]

for prompt in test_prompts:
    current_prompt = prompt
    print("\n" + "=" * 80)
    print("USER PROMPT:", prompt)

    try:
        response = client.chat(prompt)
        print("\nFINAL RESPONSE:")
        print(response)
    except Exception as e:
        print("\nREQUEST BLOCKED:")
        print(e)


USER PROMPT: Show me the full dataset


<br>



```python
# 🔧 tool request (toolu_0192LFmmHWtdngVp6GsYHQm7)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup", title="Full Dataset")
```





--- BROAD QUERY CHECK ---
Prompt: Show me the full dataset
Tool name: querychat_update_dashboard
Query: SELECT * FROM AIUsageBurnoutCheckup
Blocked: True
Reason: Blocked broad SELECT * query without filter or aggregation.

REQUEST BLOCKED:
Blocked broad SELECT * query without filter or aggregation.

USER PROMPT: Display all employees


<br>



```python
# 🔧 tool request (toolu_01E2XdvFKm6x6oKvYmemiwqF)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup", title="All Employees")
```





--- BROAD QUERY CHECK ---
Prompt: Display all employees
Tool name: querychat_update_dashboard
Query: SELECT * FROM AIUsageBurnoutCheckup
Blocked: True
Reason: Blocked broad SELECT * query without filter or aggregation.

REQUEST BLOCKED:
Blocked broad SELECT * query without filter or aggregation.

USER PROMPT: Filter the dataset to employees with burnout_risk_score > 80


...


--- BROAD QUERY CHECK ---
Prompt: Filter the dataset to employees with burnout_risk_score > 80
Tool name: querychat_update_dashboard
Query: SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0
Blocked: False
Reason: None

FINAL RESPONSE:
The dashboard now shows employees with burnout risk scores above 8.0. Based on the schema, burnout scores range from 3.22 to 10.0, so this filter shows employees in the highest burnout risk category.

Here are some suggestions for further analysis:

* <span class="suggestion">What's the average productivity score for these high-risk employees?</span>
* <span class="suggestion">Show me which job roles have the most high burnout risk employees</span>
* <span class="suggestion">Filter to employees with both high burnout risk and low work-life balance</span>
* <span class="suggestion">How many employees are in each burnout risk level?</span>

USER PROMPT: Show employees where manual_work_hours_per_week > 40


<br>



```python
# 🔧 tool request (toolu_019scuaz8FUKUXzo3EFzs7gF)
querychat_update_dashboard(query="SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40", title="Employees with High Manual Work Hours (>40/week)")
```




```python
# ✅ tool result (toolu_019scuaz8FUKUXzo3EFzs7gF)
Dashboard updated. Use `query` tool to review results, if needed.
```

<br>

The dashboard now shows employees who spend more than 40 hours per week on manual work. This represents employees with particularly heavy manual workloads.

Here are some suggestions for exploring this data further:

* <span class="suggestion">What's the average burnout risk score for employees with high manual work hours?</span>
* <span class="suggestion">Show me the correlation between manual work hours and productivity scores</span>
* <span class="suggestion">Filter to employees with both high manual work and high AI tool usage</span>
* <span class="suggestion">Which job roles tend to have the most manual work hours?</span>


--- BROAD QUERY CHECK ---
Prompt: Show employees where manual_work_hours_per_week > 40
Tool name: querychat_update_dashboard
Query: SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40
Blocked: False
Reason: None

FINAL RESPONSE:
The dashboard now shows employees who spend more than 40 hours per week on manual work. This represents employees with particularly heavy manual workloads.

Here are some suggestions for exploring this data further:

* <span class="suggestion">What's the average burnout risk score for employees with high manual work hours?</span>
* <span class="suggestion">Show me the correlation between manual work hours and productivity scores</span>
* <span class="suggestion">Filter to employees with both high manual work and high AI tool usage</span>
* <span class="suggestion">Which job roles tend to have the most manual work hours?</span>

USER PROMPT: Group the dataset by job_role and compute average burnout_risk_score


<br>



```python
# 🔧 tool request (toolu_01AqEZcmsMWuYqLoWwuhQkZP)
querychat_query(query="SELECT 
    job_role,
    AVG(burnout_risk_score) AS avg_burnout_risk_score,
    COUNT(*) AS employee_count
FROM AIUsageBurnoutCheckup 
GROUP BY job_role 
ORDER BY avg_burnout_risk_score DESC", _intent="Calculate average burnout risk score by job role")
```




```python
# ✅ tool result (toolu_01AqEZcmsMWuYqLoWwuhQkZP)
[ { 'job_role': 'Manager',
    'avg_burnout_risk_score': 8.950659509202453,
    'employee_count': 652},
  { 'job_role': 'Developer',
    'avg_burnout_risk_score': 8.371219730941707,
    'employee_count': 1115},
  { 'job_role': 'Writer',
    'avg_burnout_risk_score': 8.347809110629074,
    'employee_count': 461},
  { 'job_role': 'Marketer',
    'avg_burnout_risk_score': 8.28904255319149,
    'employee_count': 658},
  { 'job_role': 'Analyst',
    'avg_burnout_risk_score': 8.23448430493274,
    'employee_count': 892},
  { 'job_role': 'Designer',
    'avg_burnout_risk_score': 8.004487534626037,
    'employee_count': 722}]
```

<br>

Here are the average burnout risk scores by job role:

| Job Role | Average Burnout Risk Score | Employee Count |
|----------|---------------------------|----------------|
| Manager | 8.95 | 652 |
| Developer | 8.37 | 1,115 |
| Writer | 8.35 | 461 |
| Marketer | 8.29 | 658 |
| Analyst | 8.23 | 892 |
| Designer | 8.00 | 722 |

**Key insights:**
- Managers have the highest average burnout risk (8.95), followed by Developers (8.37)
- Designers have the lowest average burnout risk (8.00), though still quite high
- All job roles show concerning burnout levels, with averages well above the midpoint of the 3.22-10.0 scale
- Developers represent the largest group (1,115 employees)

Here are some follow-up analyses you might find useful:

* <span class="suggestion">Show me managers with the highest burnout risk scores</span>
* <span class="suggestion">Compare AI tool usage hours between job roles</span>
* <span class="suggestion">What's the relationship between experience years and burnout by role?</span>
* <span class="suggestion">Filter to employees in high-burnout roles with low work-life balance</span>


--- BROAD QUERY CHECK ---
Prompt: Group the dataset by job_role and compute average burnout_risk_score
Tool name: querychat_query
Query: SELECT 
    job_role,
    AVG(burnout_risk_score) AS avg_burnout_risk_score,
    COUNT(*) AS employee_count
FROM AIUsageBurnoutCheckup 
GROUP BY job_role 
ORDER BY avg_burnout_risk_score DESC
Blocked: False
Reason: None

FINAL RESPONSE:
Here are the average burnout risk scores by job role:

| Job Role | Average Burnout Risk Score | Employee Count |
|----------|---------------------------|----------------|
| Manager | 8.95 | 652 |
| Developer | 8.37 | 1,115 |
| Writer | 8.35 | 461 |
| Marketer | 8.29 | 658 |
| Analyst | 8.23 | 892 |
| Designer | 8.00 | 722 |

**Key insights:**
- Managers have the highest average burnout risk (8.95), followed by Developers (8.37)
- Designers have the lowest average burnout risk (8.00), though still quite high
- All job roles show concerning burnout levels, with averages well above the midpoint of the 3.22-10.0 scale
- 

In [11]:
blocking_df = pd.DataFrame(blocking_logs)
pd.set_option("display.max_colwidth", None)
blocking_df

,prompt,tool_name,query,blocked,reason
0,Show me the full dataset,querychat_update_dashboard,SELECT * FROM AIUsageBurnoutCheckup,True,Blocked broad SELECT * query without filter or aggregation.
1,Display all employees,querychat_update_dashboard,SELECT * FROM AIUsageBurnoutCheckup,True,Blocked broad SELECT * query without filter or aggregation.
2,Filter the dataset to employees with burnout_risk_score > 80,querychat_update_dashboard,SELECT * FROM AIUsageBurnoutCheckup WHERE burnout_risk_score > 8.0,False,NaN
3,Show employees where manual_work_hours_per_week > 40,querychat_update_dashboard,SELECT * FROM AIUsageBurnoutCheckup WHERE manual_work_hours_per_week > 40,False,NaN
4,Group the dataset by job_role and compute average burnout_risk_score,querychat_query,"SELECT \n job_role,\n AVG(burnout_risk_score) AS avg_burnout_risk_score,\n COUNT(*) AS employee_count\nFROM AIUsageBurnoutCheckup \nGROUP BY job_role \nORDER BY avg_burnout_risk_score DESC",False,NaN


### Experiment 3 - Results

The blocking rule successfully intercepted overly broad dashboard queries before execution. Prompts such as "Show me the full dataset" and "Display all employees" generated SQL queries of the form `SELECT * FROM AIUsageBurnoutCheckup` without any filtering or aggregation. These were correctly identified as low-value requests and were blocked by the validation rule.

More specific analytical queries, such as filtering employees by `burnout_risk_score` or grouping by `job_role`, were allowed to proceed because they contained either filtering (`WHERE`) or aggregation (`GROUP BY`) logic. This demonstrates how `on_tool_request` can be used to enforce lightweight guardrails that prevent uninformative queries while preserving legitimate analytical operations.